## NOTE: this notebook was made for colab to save the ouput to personal drive

In [4]:
!pip -q install umap-learn
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np, pandas as pd, umap
from pathlib import Path
from sklearn.cluster import KMeans, AgglomerativeClustering, HDBSCAN
from sklearn.metrics import (adjusted_rand_score as ARI, normalized_mutual_info_score as NMI,
                             silhouette_score, davies_bouldin_score)

CACHE = Path("/content/drive/MyDrive/legal_idp/embeddings"); SEED = 42
sp = pd.read_csv(CACHE / "split.csv")                # upload next to the notebook
assert (sp["id"].to_numpy() == np.load(CACHE / "ids.npy")).all()
purity = lambda y, c: pd.crosstab(c, y).max(1).sum() / len(y)

rows = []
for setup, mask in [("18 classes", (sp["label"] != "Unknown").to_numpy()), ("19 classes", np.ones(len(sp), bool))]:
    y = sp.loc[mask, "label"].to_numpy(); k = len(np.unique(y))
    for name, file in [("siglip", "siglip_siglip-base-patch16-224.npy"), ("jina", "jina_v4.npy")]:
        X = np.load(CACHE / file); X = X / np.linalg.norm(X, axis=1, keepdims=True); Xm = X[mask]
        Z = umap.UMAP(n_components=10, metric="cosine", random_state=SEED).fit_transform(Xm)
        for algo, c in {"kmeans": KMeans(k, n_init=10, random_state=SEED).fit_predict(Xm),
                        "agglomerative": AgglomerativeClustering(k, linkage="ward").fit_predict(Xm),
                        "hdbscan": HDBSCAN(min_cluster_size=10).fit_predict(Z)}.items():
            ok = c != -1; n = len(set(c[ok]))
            rows.append(dict(setup=setup, model=name, algo=algo, clusters=n, noise_pct=round((~ok).mean()*100, 1),
                             ARI=ARI(y, c), NMI=NMI(y, c), purity=purity(y, c),
                             silhouette=silhouette_score(Xm[ok], c[ok], metric="cosine") if n > 1 else np.nan,
                             DB=davies_bouldin_score(Xm[ok], c[ok]) if n > 1 else np.nan))
pd.DataFrame(rows).round(3)

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,setup,model,algo,clusters,noise_pct,ARI,NMI,purity,silhouette,DB
0,18 classes,siglip,kmeans,18,0.0,0.884,0.941,0.919,0.392,1.889
1,18 classes,siglip,agglomerative,18,0.0,0.895,0.944,0.952,0.388,1.824
2,18 classes,siglip,hdbscan,20,1.5,0.869,0.938,0.925,0.382,1.712
3,18 classes,jina,kmeans,18,0.0,0.813,0.896,0.884,0.603,1.072
4,18 classes,jina,agglomerative,18,0.0,0.813,0.910,0.885,0.583,1.137
5,18 classes,jina,hdbscan,26,0.5,0.770,0.904,0.925,0.588,1.084
6,19 classes,siglip,kmeans,19,0.0,0.808,0.901,0.886,0.348,1.949
7,19 classes,siglip,agglomerative,19,0.0,0.839,0.920,0.922,0.343,2.070
8,19 classes,siglip,hdbscan,20,0.6,0.693,0.873,0.827,0.325,1.901
9,19 classes,jina,kmeans,19,0.0,0.719,0.845,0.828,0.536,1.163


In [6]:
pd.DataFrame(rows).round(3).to_csv(CACHE / "clustering_results.csv", index=False)